In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

from scipy import stats
from scipy.stats import wilcoxon

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

from tqdm.auto import tqdm


warnings.filterwarnings("ignore")


# ============================================================
# 路径
# ============================================================

DATA_PATH = Path(
    "./data/voc_dataset_1+2_vs_3.mat"
)

OUTPUT_DIR = Path(
    "./result/topk_classical_benchmark"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# 当前 MultiView 正式基线的逐次结果
BASELINE_REPEAT_PATH = Path(
    "./result/1+2_vs_3_lambda_001_formal/"
    "metrics_per_repeat.csv"
)


# ============================================================
# 实验参数
# ============================================================

TOP_K_LIST = [
    20,
    50,
    100,
    200,
]

MODEL_NAMES = [
    "LogisticRegression",
    "LinearSVM",
]

OUTER_REPEATS = 30
TEST_SIZE = 0.20
SEED = 42

INNER_FOLDS = 5

C_VALUES = [
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
]


print("数据文件：", DATA_PATH.resolve())
print("结果目录：", OUTPUT_DIR.resolve())
print("外层重复次数：", OUTER_REPEATS)
print("Top-K：", TOP_K_LIST)
print("模型：", MODEL_NAMES)

In [ ]:
def decode_matlab_string(value):
    while (
        isinstance(value, np.ndarray)
        and value.size == 1
    ):
        value = value.reshape(-1)[0]

    if isinstance(value, bytes):
        return value.decode(
            "utf-8",
            errors="replace",
        ).strip()

    if isinstance(value, np.ndarray):
        if value.dtype.kind in {"U", "S"}:
            return "".join(
                value.astype(str).reshape(-1)
            ).strip()

        return str(
            value.squeeze()
        ).strip()

    return str(value).strip()


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"找不到数据文件：{DATA_PATH.resolve()}"
    )


mat_data = sio.loadmat(
    DATA_PATH
)

if "X" not in mat_data or "y" not in mat_data:
    raise KeyError(
        "MAT 文件中必须包含 X 和 y。"
    )


X = np.asarray(
    mat_data["X"],
    dtype=np.float64,
)

y = np.asarray(
    mat_data["y"],
    dtype=np.int64,
).reshape(-1)


if "feat_names" in mat_data:
    feature_names = [
        decode_matlab_string(value)
        for value in mat_data[
            "feat_names"
        ].reshape(-1)
    ]
else:
    feature_names = [
        f"VOC_{index}"
        for index in range(
            X.shape[1]
        )
    ]


if len(feature_names) != X.shape[1]:
    feature_names = [
        f"VOC_{index}"
        for index in range(
            X.shape[1]
        )
    ]


if X.ndim != 2:
    raise ValueError(
        f"X 必须是二维矩阵，当前为 {X.shape}"
    )

if len(y) != X.shape[0]:
    raise ValueError(
        "X 的样本数和 y 的标签数不一致。"
    )

if not np.isfinite(X).all():
    raise ValueError(
        "X 中存在 NaN 或 Inf。"
    )

if set(np.unique(y)) != {0, 1}:
    raise ValueError(
        f"标签必须为 0/1，当前为 {np.unique(y)}"
    )


print("数据读取完成")
print("-" * 60)
print("样本数：", X.shape[0])
print("特征数：", X.shape[1])
print(
    "类别数量：",
    dict(
        zip(
            *np.unique(
                y,
                return_counts=True,
            )
        )
    ),
)

In [ ]:
def calculate_metrics(
    y_true,
    y_pred,
    decision_scores,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    decision_scores = np.asarray(
        decision_scores,
        dtype=float,
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = cm.ravel()

    eps = 1e-12

    sensitivity = tp / (
        tp + fn + eps
    )

    specificity = tn / (
        tn + fp + eps
    )

    ppv = tp / (
        tp + fp + eps
    )

    npv = tn / (
        tn + fn + eps
    )

    try:
        auc_value = roc_auc_score(
            y_true,
            decision_scores,
        )
    except ValueError:
        auc_value = np.nan

    return {
        "Sensitivity": float(sensitivity),
        "Specificity": float(specificity),
        "PPV": float(ppv),
        "NPV": float(npv),

        "Accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "Balanced_Accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "F1": float(
            f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),

        "AUC": float(auc_value),
    }


def mean_ci(
    values,
    alpha=0.05,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    n = len(values)

    if n == 0:
        return (
            np.nan,
            np.nan,
            np.nan,
            np.nan,
        )

    mean_value = float(
        values.mean()
    )

    if n < 2:
        return (
            mean_value,
            np.nan,
            np.nan,
            np.nan,
        )

    std_value = float(
        values.std(ddof=1)
    )

    sem_value = (
        std_value / np.sqrt(n)
    )

    t_value = stats.t.ppf(
        1 - alpha / 2,
        df=n - 1,
    )

    lower = (
        mean_value
        - t_value * sem_value
    )

    upper = (
        mean_value
        + t_value * sem_value
    )

    return (
        mean_value,
        std_value,
        lower,
        upper,
    )


def select_training_threshold(
    y_true,
    scores,
    model_name,
):
    """
    只根据外层训练集的内层交叉验证预测选择阈值。

    优化目标：
        首先最大化 min(Accuracy, F1)，
        避免仅提高其中一个指标。
    """

    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    scores = np.asarray(
        scores,
        dtype=float,
    )


    if model_name == "LogisticRegression":
        low = max(
            0.01,
            float(
                np.quantile(
                    scores,
                    0.01,
                )
            ),
        )

        high = min(
            0.99,
            float(
                np.quantile(
                    scores,
                    0.99,
                )
            ),
        )

        default_threshold = 0.5

    else:
        low = float(
            np.quantile(
                scores,
                0.01,
            )
        )

        high = float(
            np.quantile(
                scores,
                0.99,
            )
        )

        default_threshold = 0.0


    if high <= low:
        thresholds = np.asarray(
            [default_threshold]
        )

    else:
        thresholds = np.unique(
            np.concatenate(
                [
                    np.linspace(
                        low,
                        high,
                        181,
                    ),

                    np.asarray(
                        [default_threshold]
                    ),
                ]
            )
        )


    best_result = None


    for threshold in thresholds:
        predictions = (
            scores >= threshold
        ).astype(int)

        accuracy = accuracy_score(
            y_true,
            predictions,
        )

        f1 = f1_score(
            y_true,
            predictions,
            zero_division=0,
        )

        balanced_accuracy = (
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        )

        joint_score = min(
            accuracy,
            f1,
        )

        secondary_score = (
            accuracy
            + f1
            + balanced_accuracy
        ) / 3.0


        candidate = {
            "Threshold": float(threshold),
            "Joint_Score": float(joint_score),
            "Secondary_Score": float(
                secondary_score
            ),
            "Accuracy": float(accuracy),
            "F1": float(f1),
        }


        if best_result is None:
            best_result = candidate
            continue


        if (
            candidate["Joint_Score"]
            > best_result["Joint_Score"]
            + 1e-12
        ):
            best_result = candidate

        elif np.isclose(
            candidate["Joint_Score"],
            best_result["Joint_Score"],
        ):
            if (
                candidate["Secondary_Score"]
                > best_result[
                    "Secondary_Score"
                ]
            ):
                best_result = candidate


    return best_result

In [ ]:
def build_pipeline(
    model_name,
    top_k,
    random_state,
):
    if model_name == "LogisticRegression":
        classifier = LogisticRegression(
            class_weight="balanced",
            solver="liblinear",
            max_iter=5000,
            random_state=random_state,
        )

    elif model_name == "LinearSVM":
        classifier = LinearSVC(
            class_weight="balanced",
            max_iter=50000,

            # 兼容较旧的 sklearn 版本
            dual=True,

            random_state=random_state,
        )

    else:
        raise ValueError(
            f"不支持的模型：{model_name}"
        )


    pipeline = Pipeline(
        steps=[
            # 每个训练折内部重新进行特征排序
            (
                "feature_selection",
                SelectKBest(
                    score_func=f_classif,
                    k=top_k,
                ),
            ),

            # 每个训练折内部重新计算标准化参数
            (
                "standard_scaler",
                StandardScaler(),
            ),

            (
                "classifier",
                classifier,
            ),
        ]
    )


    parameter_grid = {
        "classifier__C": C_VALUES,
    }


    return (
        pipeline,
        parameter_grid,
    )

In [ ]:
all_indices = np.arange(
    X.shape[0]
)


all_rows = []


selection_counts = {
    (model_name, top_k): np.zeros(
        X.shape[1],
        dtype=int,
    )
    for model_name in MODEL_NAMES
    for top_k in TOP_K_LIST
}


total_tasks = (
    OUTER_REPEATS
    * len(TOP_K_LIST)
    * len(MODEL_NAMES)
)


progress = tqdm(
    total=total_tasks,
    desc="Top-K benchmark",
)


for repeat_index in range(
    OUTER_REPEATS
):
    # 与现有 MultiView 正式实验使用相同外层种子
    outer_seed = (
        SEED
        + repeat_index * 1000
    )


    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=TEST_SIZE,
        random_state=outer_seed,
        stratify=y,
    )


    X_train = X[
        train_indices
    ]

    y_train = y[
        train_indices
    ]

    X_test = X[
        test_indices
    ]

    y_test = y[
        test_indices
    ]


    inner_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=outer_seed + 1,
    )


    for top_k in TOP_K_LIST:
        actual_k = min(
            top_k,
            X.shape[1],
        )


        for model_name in MODEL_NAMES:
            pipeline, parameter_grid = (
                build_pipeline(
                    model_name=model_name,
                    top_k=actual_k,
                    random_state=outer_seed,
                )
            )


            # ================================================
            # 内层交叉验证选择 C
            # ================================================

            search = GridSearchCV(
                estimator=pipeline,
                param_grid=parameter_grid,
                scoring="f1",
                cv=inner_cv,
                n_jobs=-1,
                refit=True,
                error_score="raise",
            )


            search.fit(
                X_train,
                y_train,
            )


            best_model = (
                search.best_estimator_
            )


            # ================================================
            # 仅利用训练集 OOF 预测选择分类阈值
            # ================================================

            if (
                model_name
                == "LogisticRegression"
            ):
                oof_scores = cross_val_predict(
                    best_model,
                    X_train,
                    y_train,
                    cv=inner_cv,
                    method="predict_proba",
                    n_jobs=-1,
                )[:, 1]

            else:
                oof_scores = cross_val_predict(
                    best_model,
                    X_train,
                    y_train,
                    cv=inner_cv,
                    method="decision_function",
                    n_jobs=-1,
                )


            threshold_result = (
                select_training_threshold(
                    y_true=y_train,
                    scores=oof_scores,
                    model_name=model_name,
                )
            )


            best_threshold = (
                threshold_result[
                    "Threshold"
                ]
            )


            # ================================================
            # 在完全独立的外层测试集上评估
            # ================================================

            if (
                model_name
                == "LogisticRegression"
            ):
                test_scores = (
                    best_model.predict_proba(
                        X_test
                    )[:, 1]
                )

            else:
                test_scores = (
                    best_model.decision_function(
                        X_test
                    )
                )


            test_predictions = (
                test_scores
                >= best_threshold
            ).astype(int)


            metrics = calculate_metrics(
                y_true=y_test,
                y_pred=test_predictions,
                decision_scores=test_scores,
            )


            # ================================================
            # 保存该次训练选择的特征
            # ================================================

            selector = (
                best_model.named_steps[
                    "feature_selection"
                ]
            )


            selected_indices = np.flatnonzero(
                selector.get_support()
            )


            selection_counts[
                (model_name, top_k)
            ][selected_indices] += 1


            row = {
                "Repeat": repeat_index,
                "Outer_Seed": outer_seed,
                "Model": model_name,
                "Top_K": top_k,

                "Best_C": search.best_params_[
                    "classifier__C"
                ],

                "Inner_Best_F1": float(
                    search.best_score_
                ),

                "Threshold": float(
                    best_threshold
                ),

                "Inner_Threshold_Accuracy":
                    threshold_result[
                        "Accuracy"
                    ],

                "Inner_Threshold_F1":
                    threshold_result[
                        "F1"
                    ],

                "Selected_Features": int(
                    len(selected_indices)
                ),
            }


            row.update(
                metrics
            )


            all_rows.append(
                row
            )


            progress.update(1)

            progress.set_postfix(
                {
                    "repeat": repeat_index + 1,
                    "model": model_name,
                    "K": top_k,
                    "ACC": (
                        f"{metrics['Accuracy']:.3f}"
                    ),
                    "F1": (
                        f"{metrics['F1']:.3f}"
                    ),
                }
            )


progress.close()

print("全部 Top-K 实验完成。")

In [ ]:
per_repeat_df = pd.DataFrame(
    all_rows
)


per_repeat_path = (
    OUTPUT_DIR
    / "topk_classical_per_repeat.csv"
)


per_repeat_df.to_csv(
    per_repeat_path,
    index=False,
    encoding="utf-8-sig",
)


metric_names = [
    "Accuracy",
    "F1",
    "AUC",
    "Balanced_Accuracy",
    "Sensitivity",
    "Specificity",
    "PPV",
    "NPV",
]


summary_rows = []


for (
    model_name,
    top_k,
), group_df in per_repeat_df.groupby(
    [
        "Model",
        "Top_K",
    ]
):
    row = {
        "Model": model_name,
        "Top_K": int(top_k),
    }


    for metric_name in metric_names:
        (
            mean_value,
            std_value,
            lower,
            upper,
        ) = mean_ci(
            group_df[
                metric_name
            ].values
        )


        row[
            f"{metric_name}_Mean"
        ] = mean_value

        row[
            f"{metric_name}_Std"
        ] = std_value

        row[
            f"{metric_name}_CI95_Lower"
        ] = lower

        row[
            f"{metric_name}_CI95_Upper"
        ] = upper


    row["Joint_Mean"] = min(
        row["Accuracy_Mean"],
        row["F1_Mean"],
    )


    best_c_modes = (
        group_df["Best_C"].mode()
    )

    if len(best_c_modes) > 0:
        row["Best_C_Mode"] = (
            best_c_modes.iloc[0]
        )
    else:
        row["Best_C_Mode"] = np.nan


    summary_rows.append(
        row
    )


summary_df = pd.DataFrame(
    summary_rows
)


summary_df = summary_df.sort_values(
    by=[
        "Joint_Mean",
        "F1_Mean",
        "Accuracy_Mean",
        "AUC_Mean",
    ],
    ascending=False,
).reset_index(drop=True)


summary_path = (
    OUTPUT_DIR
    / "topk_classical_summary.csv"
)


summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)


display(
    summary_df[
        [
            "Model",
            "Top_K",
            "Accuracy_Mean",
            "Accuracy_Std",
            "F1_Mean",
            "F1_Std",
            "AUC_Mean",
            "AUC_Std",
            "Balanced_Accuracy_Mean",
            "Sensitivity_Mean",
            "Specificity_Mean",
            "Joint_Mean",
            "Best_C_Mode",
        ]
    ]
)


print(
    "逐次结果：",
    per_repeat_path.resolve(),
)

print(
    "汇总结果：",
    summary_path.resolve(),
)

In [ ]:
feature_frequency_rows = []


for (
    model_name,
    top_k,
), counts in selection_counts.items():

    frequencies = (
        counts / OUTER_REPEATS
    )


    for feature_index in range(
        X.shape[1]
    ):
        feature_frequency_rows.append(
            {
                "Model": model_name,
                "Top_K": top_k,

                "VOC_Index": feature_index,

                "VOC_Name": feature_names[
                    feature_index
                ],

                "Selection_Count": int(
                    counts[
                        feature_index
                    ]
                ),

                "Selection_Frequency": float(
                    frequencies[
                        feature_index
                    ]
                ),
            }
        )


feature_frequency_df = pd.DataFrame(
    feature_frequency_rows
)


feature_frequency_path = (
    OUTPUT_DIR
    / "topk_feature_selection_frequency.csv"
)


feature_frequency_df.to_csv(
    feature_frequency_path,
    index=False,
    encoding="utf-8-sig",
)


display(
    feature_frequency_df.sort_values(
        by=[
            "Model",
            "Top_K",
            "Selection_Frequency",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    ).head(30)
)


print(
    "特征频率文件：",
    feature_frequency_path.resolve(),
)

In [ ]:
if not BASELINE_REPEAT_PATH.exists():
    raise FileNotFoundError(
        "找不到 MultiView 正式基线结果："
        f"{BASELINE_REPEAT_PATH.resolve()}"
    )


baseline_repeat_df = pd.read_csv(
    BASELINE_REPEAT_PATH
)


comparison_rows = []


# 当前 MultiView 正式基线
comparison_rows.append(
    {
        "Model": "MultiView_Baseline",
        "Top_K": "All/Gated",

        "Accuracy_Mean": (
            baseline_repeat_df[
                "Accuracy"
            ].mean()
        ),

        "F1_Mean": (
            baseline_repeat_df[
                "F1"
            ].mean()
        ),

        "AUC_Mean": (
            baseline_repeat_df[
                "AUC"
            ].mean()
        ),

        "Joint_Mean": min(
            baseline_repeat_df[
                "Accuracy"
            ].mean(),

            baseline_repeat_df[
                "F1"
            ].mean(),
        ),
    }
)


# 所有传统模型
for _, row in summary_df.iterrows():
    comparison_rows.append(
        {
            "Model": row["Model"],
            "Top_K": int(
                row["Top_K"]
            ),

            "Accuracy_Mean": row[
                "Accuracy_Mean"
            ],

            "F1_Mean": row[
                "F1_Mean"
            ],

            "AUC_Mean": row[
                "AUC_Mean"
            ],

            "Joint_Mean": row[
                "Joint_Mean"
            ],
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


comparison_df = comparison_df.sort_values(
    by=[
        "Joint_Mean",
        "F1_Mean",
        "Accuracy_Mean",
    ],
    ascending=False,
).reset_index(drop=True)


comparison_path = (
    OUTPUT_DIR
    / "topk_vs_multiview_comparison.csv"
)


comparison_df.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig",
)


display(
    comparison_df
)


labels = [
    f"{model}\n{top_k}"
    for model, top_k in zip(
        comparison_df["Model"],
        comparison_df["Top_K"],
    )
]


x_positions = np.arange(
    len(comparison_df)
)


bar_width = 0.25


figure, axis = plt.subplots(
    figsize=(15, 7)
)


axis.bar(
    x_positions - bar_width,
    comparison_df[
        "Accuracy_Mean"
    ],
    width=bar_width,
    label="Accuracy",
)


axis.bar(
    x_positions,
    comparison_df[
        "F1_Mean"
    ],
    width=bar_width,
    label="F1",
)


axis.bar(
    x_positions + bar_width,
    comparison_df[
        "AUC_Mean"
    ],
    width=bar_width,
    label="AUC",
)


axis.axhline(
    0.80,
    linestyle="--",
    linewidth=1.5,
    label="Target = 0.80",
)


axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    labels,
    rotation=45,
    ha="right",
)

axis.set_ylim(
    0,
    1.0,
)

axis.set_ylabel(
    "Mean outer-test performance"
)

axis.set_title(
    "Leakage-safe Top-K benchmark vs MultiView"
)

axis.legend()


figure.tight_layout()


figure_path = (
    OUTPUT_DIR
    / "topk_vs_multiview_comparison.png"
)


figure.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)


plt.show()


print(
    "比较表：",
    comparison_path.resolve(),
)

print(
    "比较图：",
    figure_path.resolve(),
)

In [ ]:
best_classical = summary_df.iloc[0]


best_model_name = best_classical[
    "Model"
]

best_top_k = int(
    best_classical["Top_K"]
)


best_repeat_df = per_repeat_df[
    (
        per_repeat_df["Model"]
        == best_model_name
    )
    &
    (
        per_repeat_df["Top_K"]
        == best_top_k
    )
].copy()


paired_data = pd.merge(
    baseline_repeat_df[
        [
            "Repeat",
            "Accuracy",
            "F1",
            "AUC",
        ]
    ],

    best_repeat_df[
        [
            "Repeat",
            "Accuracy",
            "F1",
            "AUC",
        ]
    ],

    on="Repeat",

    suffixes=(
        "_MultiView",
        "_Classical",
    ),
)


paired_rows = []


for metric_name in [
    "Accuracy",
    "F1",
    "AUC",
]:
    baseline_values = paired_data[
        f"{metric_name}_MultiView"
    ].to_numpy(dtype=float)


    classical_values = paired_data[
        f"{metric_name}_Classical"
    ].to_numpy(dtype=float)


    differences = (
        classical_values
        - baseline_values
    )


    try:
        statistic, p_value = wilcoxon(
            classical_values,
            baseline_values,
            zero_method="wilcox",
            alternative="two-sided",
        )

    except ValueError:
        statistic = np.nan
        p_value = np.nan


    paired_rows.append(
        {
            "Metric": metric_name,

            "MultiView_Mean": float(
                baseline_values.mean()
            ),

            "Classical_Mean": float(
                classical_values.mean()
            ),

            "Classical_Minus_MultiView":
                float(
                    differences.mean()
                ),

            "Wilcoxon_Statistic":
                statistic,

            "P_Value": p_value,
        }
    )


paired_test_df = pd.DataFrame(
    paired_rows
)


paired_test_path = (
    OUTPUT_DIR
    / "best_classical_vs_multiview_paired.csv"
)


paired_test_df.to_csv(
    paired_test_path,
    index=False,
    encoding="utf-8-sig",
)


print("=" * 70)
print("最佳传统模型")
print("-" * 70)
print("模型：", best_model_name)
print("Top-K：", best_top_k)

print(
    "Accuracy：",
    f"{best_classical['Accuracy_Mean']:.3f}",
)

print(
    "F1：",
    f"{best_classical['F1_Mean']:.3f}",
)

print(
    "AUC：",
    f"{best_classical['AUC_Mean']:.3f}",
)

print("=" * 70)


display(
    paired_test_df
)


print(
    "配对检验文件：",
    paired_test_path.resolve(),
)